In [1]:
import html
import pathlib
import re
import zipfile
import xml.etree.ElementTree as ET

def createCleanedText(filePath):
    docxPath = pathlib.Path(filePath)
    ns = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}

    with zipfile.ZipFile(docxPath, 'r') as docx:
        settingsData = docx.read('word/settings.xml').decode('utf-8')

    root = ET.fromstring(settingsData)

    docVars = root.findall(".//w:docVars/w:docVar", ns)

    package_entries = []
    for docvar in docVars:
        name = docvar.get("{http://schemas.openxmlformats.org/wordprocessingml/2006/main}name")
        if name and name.startswith("DOCDRAFTERPACKAGE_"):
            value = docvar.get("{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val")
            if value:
                package_entries.append((name, value))

    if not package_entries:
        raise ValueError("No DOCDRAFTERPACKAGE_* values were found in word/settings.xml")

    package_entries.sort(key=lambda item: int(item[0].split("_")[-1]) if item[0].split("_")[-1].isdigit() else -1)

    cleaned_parts = []
    for package_name, package_value in package_entries:
        cleaned = package_value.replace("_x00d__x00a_", "\n")
        cleaned = cleaned.replace("_x000d_", "\n")
        cleaned = cleaned.replace("_x000a_", "\n")
        cleaned = cleaned.replace("\r\n", "\n").replace("\r", "\n")
        cleaned = html.unescape(cleaned)

        if cleaned.startswith("<?xml"):
            cleaned = cleaned.split("?>", 1)[1].strip()

        cleaned_parts.append(f"<!-- {package_name} -->\n{cleaned}")

    cleaned = "\n\n".join(cleaned_parts)

    output_path = pathlib.Path('cleaned.txt')
    output_path.write_text(cleaned, encoding='utf-8')

    print("Processed", len(package_entries), "package entries")
    print("Saved cleaned XML to", output_path)
    print(cleaned)


def findIrrelevantVars(docx_path, cleaned_text_path="cleaned.txt", prefixes=None):
    docx_path = pathlib.Path(docx_path)
    cleaned_path = pathlib.Path(cleaned_text_path)

    if not cleaned_path.exists():
        createCleanedText(docx_path)

    ns = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
    prefixes = {prefix.upper() for prefix in (prefixes or ["Q", "A", "F"])}

    with zipfile.ZipFile(docx_path, 'r') as docx:
        document_xml = docx.read('word/document.xml').decode('utf-8', errors='ignore')

    root = ET.fromstring(document_xml)
    field_code_pattern = re.compile(r"(?<![A-Za-z])([A-Za-z])\s*(\d+)(?![A-Za-z0-9])")

    field_codes = []
    for alias in root.findall('.//w:alias', ns):
        text = alias.get('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val', '') or ''
        for match in field_code_pattern.finditer(text):
            prefix = match.group(1).upper()
            if prefix not in prefixes:
                continue
            number = int(match.group(2))
            field_codes.append({
                "prefix": prefix,
                "number": number,
                "token": f"{prefix}{number}",
                "context": text.strip(),
            })

    cleaned_text = cleaned_path.read_text(encoding='utf-8')
    display_pattern = re.compile(r"<(?:DisplayNum|DisplayNumber)>(\d+)</(?:DisplayNum|DisplayNumber)>", re.IGNORECASE)
    display_numbers = {int(match.group(1)) for match in display_pattern.finditer(cleaned_text)}

    missing_numbers = sorted({item['number'] for item in field_codes if item['number'] not in display_numbers})
    code_numbers = {item['number'] for item in field_codes}
    missing_display_tuples = [("Q", number) for number in sorted(display_numbers) if number not in code_numbers]

    return {
        "field_codes": field_codes,
        "display_numbers": sorted(display_numbers),
        "missing_numbers": missing_numbers,
        "missing_display_tags": sorted(display_numbers),

        "missing_display_tuples": missing_display_tuples,
    }

In [ ]:
createCleanedText("Will for Individual with Partner (Trust for Partner) (IL).docx")
result = findIrrelevantVars("Will for Individual with Partner (Trust for Partner) (IL).docx")
print("field codes found:", len(result["field_codes"]))
print("unique codes:", sorted({(item["prefix"], item["number"]) for item in result["field_codes"]}, key=lambda item: item[1]))
print("display numbers:", result["display_numbers"])
formatted_missing = ", ".join(f"{prefix}{number}" for prefix, number in result["missing_display_tuples"]) or "None"
print("missing display tuples:", formatted_missing)


Processed 10 package entries
Saved cleaned XML to cleaned.txt
<!-- DOCDRAFTERPACKAGE_0 -->
<Package xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">

  <XMLVersion>3</XMLVersion>

  <PortalUrl>https://lexisnexis.documentdrafter.com/</PortalUrl>

  <Version>1</Version>

  <Name>Will for Individual with Partner (Trust for Partner) (IL)</Name>

  <Id>98b86297-82ba-407f-abd5-f5986863b37a</Id>

  <Templates>

    <Template>

      <Name>3319710_TE_WillForIndividualWithPartner_TrustForPartner_IL.docx</Name>

      <DownloadName>[F39]</DownloadName>

      <Id>f3f0b97e-ddab-411f-ac28-71d97694af31</Id>

      <Conditions />

      <Clauses />

      <Calculations />

      <RepeatId />

      <updated>2026-05-28T13:00:06.2345168Z</updated>

      <isDirty>false</isDirty>

    </Template>

  </Templates>

  <Questions>

    <Question>

      <DisplayNumber>1</DisplayNumber>

      <Id>ac5da7e8-a11e-41c6-8e96-ebcd82b53e8a</Id>

      <LinkBitQue

In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox


def chooseDocx():
    file_path = filedialog.askopenfilename(title="Select DOCX file", filetypes=[("Word Documents", "*.docx")])
    if file_path:
        docx_path_var.set(file_path)
        output_text.delete("1.0", tk.END)


def runCleanAndFind():
    """
    Todo: Add a disclaimer to remind the user to refresh the interview first before running this function, as the cleaned text may be outdated if the interview is using incorrect display numbers.
    """
    docx_path = docx_path_var.get().strip()
    if not docx_path:
        messagebox.showwarning("No File Selected", "Please select a .docx file first.")
        return
    try:
        createCleanedText(docx_path)
        result = findIrrelevantVars(docx_path)
        unique_codes = sorted({(item["prefix"], item["number"]) for item in result["field_codes"]}, key=lambda item: item[1])
        formatted_missing = ", ".join(f"{prefix}{number}" for prefix, number in result["missing_display_tuples"]) or "None"
        output_lines = [
            f"missing display tuples: {formatted_missing}\n"
        ]
        output_text.delete("1.0", tk.END)
        output_text.insert(tk.END, "\n".join(output_lines))
    except Exception as exc:
        messagebox.showerror("Error", str(exc))


def runExplanatoryNoteSearch():
    docx_path = docx_path_var.get().strip()
    query = explanatory_query_var.get().strip()

    if not docx_path:
        messagebox.showwarning("No File Selected", "Please select a .docx file first.")
        return
    if not query:
        messagebox.showwarning("No Search Text", "Please enter text to search for in explanatory notes.")
        return

    try:
        createCleanedText(docx_path)
        explanatory_note_dict = buildExplanatoryDict("cleaned.txt")
        matches = findMatchingExplanatoryNotes(query, [explanatory_note_dict])
        formatted_matches = []
        for key in matches:
            note_text = explanatory_note_dict.get(key, "")
            formatted_matches.append(f"Question {key}:\n\n{note_text}")
        output_lines = [
            f"search query: {query}\n"
            f"matching display numbers: {matches}\n"
            f"matching note entries:\n"
            + "\n\n".join(formatted_matches)
        ]
        output_text.delete("1.0", tk.END)
        output_text.insert(tk.END, "\n".join(output_lines))
    except Exception as exc:
        messagebox.showerror("Error", str(exc))


root = tk.Tk()
root.title("Document Drafter Tool")
root.geometry("760x440")

docx_path_var = tk.StringVar()
explanatory_query_var = tk.StringVar()

main_frame = tk.Frame(root, padx=10, pady=10)
main_frame.pack(fill=tk.BOTH, expand=True)

tk.Label(main_frame, text="Selected DOCX:").grid(row=0, column=0, sticky="w")
tk.Entry(main_frame, textvariable=docx_path_var, width=70).grid(row=0, column=1, sticky="we", padx=(5, 0))
tk.Button(main_frame, text="Browse", command=chooseDocx).grid(row=0, column=2, padx=(5, 0))

tk.Button(main_frame, text="Find Irrelevant Variables", command=runCleanAndFind).grid(row=1, column=0, columnspan=3, pady=(10, 0), sticky="we")

tk.Label(main_frame, text="Search explanatory notes:").grid(row=2, column=0, sticky="w", pady=(10, 0))
tk.Entry(main_frame, textvariable=explanatory_query_var, width=70).grid(row=3, column=0, columnspan=3, sticky="we", pady=(5, 0))
tk.Button(main_frame, text="Find Matching Explanatory Notes", command=runExplanatoryNoteSearch).grid(row=4, column=0, columnspan=3, pady=(5, 0), sticky="we")

tk.Label(main_frame, text="Results:").grid(row=5, column=0, columnspan=3, sticky="w", pady=(10, 0))
output_text = tk.Text(main_frame, height=12, wrap="word")
output_text.grid(row=6, column=0, columnspan=3, sticky="nsew")
scrollbar = tk.Scrollbar(main_frame, command=output_text.yview)
scrollbar.grid(row=6, column=3, sticky="ns")
output_text.config(yscrollcommand=scrollbar.set)

main_frame.columnconfigure(1, weight=1)
main_frame.rowconfigure(6, weight=1)

root.mainloop()


Processed 2 package entries
Saved cleaned XML to cleaned.txt
<!-- DOCDRAFTERPACKAGE_0 -->
<Package xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">

  <XMLVersion>3</XMLVersion>

  <PortalUrl>https://lexisnexis.documentdrafter.com/</PortalUrl>

  <Version>5</Version>

  <Name>Unilateral Confidentiality Agreement (Short Form, Pro-Disclosing Party)</Name>

  <Id>4d2902dd-8a23-412c-bfcf-4a88e87e2c94</Id>

  <Templates>

    <Template>

      <Name>1517192_CT_UnilateralConfidentialityAgreement_ShortForm_ProDisclosingParty.docx</Name>

      <DownloadName>[F9]</DownloadName>

      <Id>84bfc809-ca03-4134-8382-47ece53f12b0</Id>

      <Conditions />

      <Clauses />

      <Calculations />

      <RepeatId />

      <updated>2026-07-01T03:51:49.9342947Z</updated>

      <isDirty>false</isDirty>

    </Template>

  </Templates>

  <Questions>

    <Question>

      <DisplayNumber>1</DisplayNumber>

      <Id>3b3eb362-b74c-441d-8f6e-496adb5f

In [2]:
from pathlib import Path
import re
from html import unescape

def buildExplanatoryDict(file_path="cleaned.txt"):
    path = Path(file_path)
    if not path.exists():
        path = Path.cwd() / file_path

    text = path.read_text(encoding="utf-8", errors="ignore")
    question_blocks = re.findall(r"<Question>(.*?)</Question>", text, re.DOTALL)
    explanatory_notes = {}

    for block in question_blocks:
        display_number_match = re.search(r"<DisplayNumber>(.*?)</DisplayNumber>", block, re.DOTALL)
        if not display_number_match:
            continue

        display_number = display_number_match.group(1).strip()
        explanatory_note_match = re.search(r"<ExplanatoryNote>(.*?)</ExplanatoryNote>", block, re.DOTALL)

        if explanatory_note_match:
            note = explanatory_note_match.group(1).strip()
            if note:
                note = re.sub(r"<[^>]+>", " ", note)
                note = re.sub(r"\s+", " ", note).strip()
                note = unescape(note)
            else:
                note = "NO EXPLANATORY NOTE"
        else:
            note = "NO EXPLANATORY NOTE"

        explanatory_notes[display_number] = note

    return explanatory_notes


explanatory_note_dict = buildExplanatoryDict()
print(f"Created {len(explanatory_note_dict)} entries.")
print(explanatory_note_dict)

Created 104 entries.
{'1': 'NO EXPLANATORY NOTE', '2': 'NO EXPLANATORY NOTE', '3': 'NO EXPLANATORY NOTE', '4': 'NO EXPLANATORY NOTE', '5': 'NO EXPLANATORY NOTE', '6': 'NO EXPLANATORY NOTE', '7': 'NO EXPLANATORY NOTE', '8': 'NO EXPLANATORY NOTE', '9': 'Drafting Note There is no legal requirement that the testator make any declaration respecting their partner, children, other descendant, or other family members. However, identifying an unmarried partner may be especially important, since such partners are not recognized under the law for inheritance purposes (unless the partners entered into a civil union). Such a declaration could bolster the surviving partner\'s claim of a partnership relationship in the event a family member challenges the nature of the relationship between the unmarried partners. Note that unlike partnerships, civil unions are legally recognized in Illinois as of the enactment of the 2011 Illinois Religious Freedom Protection and Civil Union Act (the Civil Union Act)

In [3]:
def findMatchingExplanatoryNotes(query, dictionaries=None):
    """Return the dictionary keys whose values contain the given text."""
    if dictionaries is None:
        try:
            dictionaries = [explanatory_note_dict]
        except NameError:
            raise NameError("explanatory_note_dict is not available. Run the explanatory-note setup cell first.")
    elif isinstance(dictionaries, dict):
        dictionaries = [dictionaries]

    if not isinstance(query, str):
        raise TypeError("query must be a string")

    needle = query.strip().lower()
    if not needle:
        return []

    matches = []
    for dictionary in dictionaries:
        for key, value in dictionary.items():
            if needle in str(value).lower():
                matches.append(key)

    return matches